<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fluidflow/neqsim_openfoam_compressor_inlet_wet_gas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeqSim + OpenFOAM: wet gas at a hot centrifugal-compressor inlet

This example generates a three-dimensional internal CAD volume for a centrifugal-compressor
suction line and runs a one-way NeqSim-to-OpenFOAM workflow. NeqSim supplies gas and condensate
properties. OpenFOAM calculates the turbulent carrier-gas field through a bend and contraction.
A size-resolved screening model then evaluates liquid inertia, wall impaction tendency, and the
effect of a hot compressor inlet surface.

**Learning outcomes**

- generate a watertight internal fluid volume and named STL boundary patches;
- distinguish equilibrium condensation from non-equilibrium scrubber carryover;
- transfer NeqSim density and viscosity to an OpenFOAM 14 RANS case;
- evaluate pressure loss and velocity distortion at the impeller-eye boundary;
- compare 50, 150, and 500 µm droplets using Stokes number and hot-wall evaporation sensitivity;
- identify what must be added before compressor qualification or vendor design work.


## Coupling architecture

| Stage | Model | Main outputs |
|---|---|---|
| Suction thermodynamics | NeqSim 3.16.0, SRK/classic | Phase state, density, viscosity, $C_p$, conductivity, sound speed |
| Entrained condensate | Separate NeqSim liquid composition | Liquid density, viscosity, heat capacity |
| CAD and mesh | Python STL generator + `snappyHexMesh` | `inlet`, `ductWalls`, `hotCompressorWalls`, `impellerEye` |
| Carrier flow | OpenFOAM 14, steady RANS $k$-$\epsilon$ | Velocity, pressure loss, inlet distortion |
| Liquid screening | Particle relaxation and $d^2$ evaporation | Stokes number, impaction regime, hot-contact sensitivity |

The fluid domain stops at the impeller eye. The rotating impeller and volute are not included.


## 1. Install clean-runtime dependencies

The cell installs the same pinned NeqSim and PyVista versions used by the general OpenFOAM
example. OpenFOAM 14 is installed only when it is not already available.


In [ ]:
import importlib.util
import os
from pathlib import Path
import shlex
import subprocess
import sys


NEQSIM_VERSION_REQUIRED = "3.16.0"
PYVISTA_VERSION_REQUIRED = "0.48.4"

python_requirements = {
    "neqsim": f"neqsim=={NEQSIM_VERSION_REQUIRED}",
    "pyvista": f"pyvista=={PYVISTA_VERSION_REQUIRED}",
}
missing_requirements = [
    package
    for module, package in python_requirements.items()
    if importlib.util.find_spec(module) is None
]

if missing_requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_requirements]
    )

openfoam_override = os.environ.get("OPENFOAM_BASHRC_OVERRIDE")
default_openfoam_bashrc = Path("/opt/openfoam14/etc/bashrc")

if openfoam_override and Path(openfoam_override).is_file():
    OPENFOAM_BASHRC = Path(openfoam_override)
elif default_openfoam_bashrc.is_file():
    OPENFOAM_BASHRC = default_openfoam_bashrc
else:
    install_openfoam = r"""
set -euo pipefail
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
    ca-certificates software-properties-common wget
wget -qO- https://dl.openfoam.org/gpg.key \
    > /etc/apt/trusted.gpg.d/openfoam.asc
rm -f /etc/apt/sources.list.d/*dl_openfoam_org*list
add-apt-repository -y "http://dl.openfoam.org/ubuntu main dev"
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
    --no-install-recommends openfoam14
"""
    subprocess.run(["bash", "-lc", install_openfoam], check=True)
    OPENFOAM_BASHRC = default_openfoam_bashrc

openfoam_test = (
    f"source {shlex.quote(str(OPENFOAM_BASHRC))} >/dev/null 2>&1 "
    "&& foamRun -help >/dev/null"
)
subprocess.run(["bash", "-lc", openfoam_test], check=True)

print("OpenFOAM 14 environment detected.")
print(f"Requested NeqSim version: {NEQSIM_VERSION_REQUIRED}")


In [ ]:
from importlib.metadata import version
import math
import re
import shutil

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import pyvista as pv

from neqsim import jneqsim


NEQSIM_VERSION = version("neqsim")
PYVISTA_VERSION = version("pyvista")

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

print(f"NeqSim version: {NEQSIM_VERSION}")
print(f"PyVista version: {PYVISTA_VERSION}")


## 2. Define the wet-gas engineering case

The equilibrium gas feed and the mechanically entrained liquid are separate objects. This is
important: liquid leaving a scrubber is a non-equilibrium carryover problem and should not be
replaced by the equilibrium liquid fraction from a TP flash.

The default carryover is 0.5 wt% of total flow. It is deliberately visible and editable.


In [ ]:
TEMPERATURE_K = 293.15
PRESSURE_BARA = 50.0
GAS_MASS_FLOW_KG_PER_H = 600_000.0
LIQUID_CARRYOVER_MASS_FRACTION = 0.005

gas_composition = {
    "nitrogen": 0.010,
    "CO2": 0.020,
    "methane": 0.820,
    "ethane": 0.075,
    "propane": 0.035,
    "i-butane": 0.010,
    "n-butane": 0.012,
    "i-pentane": 0.006,
    "n-pentane": 0.005,
    "n-hexane": 0.004,
    "n-heptane": 0.003,
}
liquid_composition = {
    "propane": 0.05,
    "i-butane": 0.08,
    "n-butane": 0.12,
    "i-pentane": 0.12,
    "n-pentane": 0.15,
    "n-hexane": 0.20,
    "n-heptane": 0.28,
}

assert np.isclose(sum(gas_composition.values()), 1.0, atol=1.0e-12)
assert np.isclose(sum(liquid_composition.values()), 1.0, atol=1.0e-12)


def flash_srk(composition):
    system = jneqsim.thermo.system.SystemSrkEos(TEMPERATURE_K, PRESSURE_BARA)
    for component_name, mole_fraction in composition.items():
        system.addComponent(component_name, mole_fraction)
    system.setMixingRule("classic")
    system.setMultiPhaseCheck(True)
    operations = jneqsim.thermodynamicoperations.ThermodynamicOperations(system)
    operations.TPflash()
    system.initProperties()
    system.initPhysicalProperties()
    return system


gas = flash_srk(gas_composition)
entrained_liquid = flash_srk(liquid_composition)

gas_phases = [gas.getPhase(index) for index in range(gas.getNumberOfPhases())]
gas_phase = min(gas_phases, key=lambda phase: phase.getDensity("kg/m3"))

liquid_phases = [
    entrained_liquid.getPhase(index)
    for index in range(entrained_liquid.getNumberOfPhases())
]
liquid_phase = max(
    liquid_phases,
    key=lambda phase: phase.getDensity("kg/m3"),
)

gas_density_kg_per_m3 = gas_phase.getDensity("kg/m3")
gas_viscosity_pa_s = gas_phase.getViscosity("kg/msec")
liquid_density_kg_per_m3 = liquid_phase.getDensity("kg/m3")
liquid_viscosity_pa_s = liquid_phase.getViscosity("kg/msec")
gas_mass_flow_kg_per_s = GAS_MASS_FLOW_KG_PER_H / 3600.0
liquid_mass_flow_kg_per_s = (
    gas_mass_flow_kg_per_s
    * LIQUID_CARRYOVER_MASS_FRACTION
    / (1.0 - LIQUID_CARRYOVER_MASS_FRACTION)
)

property_table = pd.DataFrame(
    {
        "gas": {
            "phase": str(gas_phase.getPhaseTypeName()),
            "density [kg/m3]": gas_density_kg_per_m3,
            "dynamic viscosity [Pa s]": gas_viscosity_pa_s,
            "Cp [J/(kg K)]": gas_phase.getCp("J/kgK"),
            "conductivity [W/(m K)]": gas_phase.getThermalConductivity("W/mK"),
            "sound speed [m/s]": gas_phase.getSoundSpeed(),
            "mass flow [kg/s]": gas_mass_flow_kg_per_s,
        },
        "entrained liquid": {
            "phase": str(liquid_phase.getPhaseTypeName()),
            "density [kg/m3]": liquid_density_kg_per_m3,
            "dynamic viscosity [Pa s]": liquid_viscosity_pa_s,
            "Cp [J/(kg K)]": liquid_phase.getCp("J/kgK"),
            "conductivity [W/(m K)]": liquid_phase.getThermalConductivity("W/mK"),
            "sound speed [m/s]": np.nan,
            "mass flow [kg/s]": liquid_mass_flow_kg_per_s,
        },
    }
)
property_table


## 3. Generate the compressor-inlet CAD

The geometry is the internal fluid volume required by CFD, not the metal casing. It contains a
600 mm suction pipe, a 90° long-radius bend, a recovery section, and a smooth contraction to a
450 mm impeller-eye interface.

The hot part of the inlet cone is a separate boundary patch so cold and hot cases can use the
same mesh.


In [ ]:
INLET_DIAMETER_M = 0.600
EYE_DIAMETER_M = 0.450
BEND_RADIUS_M = 0.900
UPSTREAM_LENGTH_M = 2.400
RECOVERY_LENGTH_M = 1.800
CONTRACTION_LENGTH_M = 0.900
EYE_DUCT_LENGTH_M = 0.300
HOT_WALL_START_FRACTION = 0.720
CIRCLE_SEGMENTS = 48


def smoothstep5(value):
    return value**3 * (10.0 - 15.0 * value + 6.0 * value**2)


def build_centerline():
    radius_inlet = 0.5 * INLET_DIAMETER_M
    radius_eye = 0.5 * EYE_DIAMETER_M
    sections = []
    radii = []

    upstream_y = np.linspace(-UPSTREAM_LENGTH_M, 0.0, 17)
    sections.append(
        np.column_stack(
            [
                np.full_like(upstream_y, -BEND_RADIUS_M),
                upstream_y,
                np.zeros_like(upstream_y),
            ]
        )
    )
    radii.append(np.full_like(upstream_y, radius_inlet))

    angle = np.linspace(math.pi, 0.5 * math.pi, 31)[1:]
    sections.append(
        np.column_stack(
            [
                BEND_RADIUS_M * np.cos(angle),
                BEND_RADIUS_M * np.sin(angle),
                np.zeros_like(angle),
            ]
        )
    )
    radii.append(np.full_like(angle, radius_inlet))

    recovery_x = np.linspace(0.0, RECOVERY_LENGTH_M, 16)[1:]
    sections.append(
        np.column_stack(
            [
                recovery_x,
                np.full_like(recovery_x, BEND_RADIUS_M),
                np.zeros_like(recovery_x),
            ]
        )
    )
    radii.append(np.full_like(recovery_x, radius_inlet))

    fraction = np.linspace(0.0, 1.0, 21)[1:]
    contraction_x = RECOVERY_LENGTH_M + CONTRACTION_LENGTH_M * fraction
    contraction_radius = (
        radius_inlet
        + (radius_eye - radius_inlet) * smoothstep5(fraction)
    )
    sections.append(
        np.column_stack(
            [
                contraction_x,
                np.full_like(contraction_x, BEND_RADIUS_M),
                np.zeros_like(contraction_x),
            ]
        )
    )
    radii.append(contraction_radius)

    eye_x = np.linspace(
        RECOVERY_LENGTH_M + CONTRACTION_LENGTH_M,
        RECOVERY_LENGTH_M + CONTRACTION_LENGTH_M + EYE_DUCT_LENGTH_M,
        6,
    )[1:]
    sections.append(
        np.column_stack(
            [
                eye_x,
                np.full_like(eye_x, BEND_RADIUS_M),
                np.zeros_like(eye_x),
            ]
        )
    )
    radii.append(np.full_like(eye_x, radius_eye))
    return np.vstack(sections), np.concatenate(radii)


def build_rings(centerline, radii):
    tangents = np.empty_like(centerline)
    tangents[0] = centerline[1] - centerline[0]
    tangents[-1] = centerline[-1] - centerline[-2]
    tangents[1:-1] = centerline[2:] - centerline[:-2]
    tangents /= np.linalg.norm(tangents, axis=1)[:, None]
    binormal = np.array([0.0, 0.0, 1.0])
    normals = np.cross(binormal[None, :], tangents)
    normals /= np.linalg.norm(normals, axis=1)[:, None]
    angles = np.linspace(0.0, 2.0 * math.pi, CIRCLE_SEGMENTS, endpoint=False)
    rings = np.empty((len(centerline), CIRCLE_SEGMENTS, 3))
    for index, (center, radius) in enumerate(zip(centerline, radii)):
        rings[index] = center + radius * (
            np.cos(angles)[:, None] * normals[index]
            + np.sin(angles)[:, None] * binormal
        )
    return rings, tangents


def build_boundaries(rings):
    ring_count = len(rings)
    duct_walls = []
    hot_walls = []
    for ring_index in range(ring_count - 1):
        fraction = (ring_index + 0.5) / (ring_count - 1)
        target = hot_walls if fraction >= HOT_WALL_START_FRACTION else duct_walls
        for point_index in range(CIRCLE_SEGMENTS):
            following = (point_index + 1) % CIRCLE_SEGMENTS
            a = rings[ring_index, point_index]
            b = rings[ring_index, following]
            c = rings[ring_index + 1, point_index]
            d = rings[ring_index + 1, following]
            target.extend([(a, b, c), (b, d, c)])

    inlet_center = rings[0].mean(axis=0)
    eye_center = rings[-1].mean(axis=0)
    inlet = []
    eye = []
    for point_index in range(CIRCLE_SEGMENTS):
        following = (point_index + 1) % CIRCLE_SEGMENTS
        inlet.append((inlet_center, rings[0, following], rings[0, point_index]))
        eye.append((eye_center, rings[-1, point_index], rings[-1, following]))
    return {
        "ductWalls": duct_walls,
        "hotCompressorWalls": hot_walls,
        "inlet": inlet,
        "impellerEye": eye,
    }


def write_stl(path, solid_name, triangles):
    with path.open("w", encoding="ascii") as stream:
        stream.write(f"solid {solid_name}\n")
        for triangle in triangles:
            first, second, third = triangle
            normal = np.cross(second - first, third - first)
            normal /= np.linalg.norm(normal)
            stream.write(
                "  facet normal "
                f"{normal[0]:.9e} {normal[1]:.9e} {normal[2]:.9e}\n"
            )
            stream.write("    outer loop\n")
            for vertex in triangle:
                stream.write(
                    "      vertex "
                    f"{vertex[0]:.9e} {vertex[1]:.9e} {vertex[2]:.9e}\n"
                )
            stream.write("    endloop\n  endfacet\n")
        stream.write(f"endsolid {solid_name}\n")


def manifold_edge_errors(boundaries):
    edge_counts = {}
    for triangles in boundaries.values():
        for triangle in triangles:
            vertices = [tuple(np.round(vertex, 8)) for vertex in triangle]
            for start, end in zip(vertices, vertices[1:] + vertices[:1]):
                edge = tuple(sorted((start, end)))
                edge_counts[edge] = edge_counts.get(edge, 0) + 1
    return sum(count != 2 for count in edge_counts.values())


centerline, radii = build_centerline()
rings, tangents = build_rings(centerline, radii)
boundaries = build_boundaries(rings)

runtime_root = Path("/content") if Path("/content").is_dir() else Path.cwd()
geometry_dir = runtime_root / "compressor_inlet_geometry"
tri_surface_dir = geometry_dir / "triSurface"
tri_surface_dir.mkdir(parents=True, exist_ok=True)

for boundary_name, triangles in boundaries.items():
    write_stl(tri_surface_dir / f"{boundary_name}.stl", boundary_name, triangles)

open_edge_count = manifold_edge_errors(boundaries)
centerline_length_m = float(
    np.linalg.norm(np.diff(centerline, axis=0), axis=1).sum()
)

assert open_edge_count == 0
assert np.allclose(tangents[0], [0.0, 1.0, 0.0], atol=1.0e-8)
assert np.allclose(tangents[-1], [1.0, 0.0, 0.0], atol=1.0e-8)

pd.Series(
    {
        "centerline length [m]": centerline_length_m,
        "triangles [-]": sum(len(value) for value in boundaries.values()),
        "open/non-manifold edges [-]": open_edge_count,
        "inlet center [m]": centerline[0],
        "impeller-eye center [m]": centerline[-1],
    },
    name="Generated CAD",
)


In [ ]:
colors = {
    "ductWalls": "#6baed6",
    "hotCompressorWalls": "#fd8d3c",
    "inlet": "#31a354",
    "impellerEye": "#de2d26",
}

figure = plt.figure(figsize=(12, 7), constrained_layout=True)
axis = figure.add_subplot(111, projection="3d")

for boundary_name, triangles in boundaries.items():
    triangle_array = np.asarray(triangles)
    stride = max(1, len(triangle_array) // 1400)
    axis.plot_trisurf(
        triangle_array[::stride, :, 0].ravel(),
        triangle_array[::stride, :, 1].ravel(),
        triangle_array[::stride, :, 2].ravel(),
        color=colors[boundary_name],
        alpha=0.72,
        linewidth=0.0,
    )

axis.plot(
    centerline[:, 0],
    centerline[:, 1],
    centerline[:, 2],
    color="black",
    linestyle="--",
    linewidth=1.4,
)
axis.set_xlabel("x [m]")
axis.set_ylabel("y [m]")
axis.set_zlabel("z [m]")
axis.set_title("Compressor-inlet internal fluid volume")
axis.view_init(elev=25, azim=-58)
axis.set_box_aspect((4.8, 4.2, 1.0))
axis.legend(
    handles=[Patch(facecolor=color, label=name) for name, color in colors.items()],
    loc="upper left",
)
plt.show()


## 4. Translate NeqSim properties to CFD inputs

The carrier solver is low-Mach and incompressible. NeqSim supplies the suction density and
kinematic viscosity $\nu=\mu/\rho$. The velocity follows from the specified mass flow and pipe
area. Turbulence quantities use a 5% inlet intensity and a length scale of $0.07D$.


In [ ]:
inlet_area_m2 = math.pi * INLET_DIAMETER_M**2 / 4.0
inlet_velocity_m_per_s = (
    gas_mass_flow_kg_per_s / (gas_density_kg_per_m3 * inlet_area_m2)
)
kinematic_viscosity_m2_per_s = gas_viscosity_pa_s / gas_density_kg_per_m3
reynolds_number = (
    inlet_velocity_m_per_s
    * INLET_DIAMETER_M
    / kinematic_viscosity_m2_per_s
)
TURBULENCE_INTENSITY = 0.05
TURBULENCE_LENGTH_SCALE_M = 0.07 * INLET_DIAMETER_M
C_MU = 0.09
turbulent_kinetic_energy_m2_per_s2 = 1.5 * (
    inlet_velocity_m_per_s * TURBULENCE_INTENSITY
) ** 2
turbulent_dissipation_m2_per_s3 = (
    C_MU**0.75
    * turbulent_kinetic_energy_m2_per_s2**1.5
    / TURBULENCE_LENGTH_SCALE_M
)
residence_time_s = centerline_length_m / inlet_velocity_m_per_s

assert reynolds_number > 4_000.0

pd.Series(
    {
        "inlet velocity [m/s]": inlet_velocity_m_per_s,
        "Reynolds number [-]": reynolds_number,
        "kinematic viscosity [m2/s]": kinematic_viscosity_m2_per_s,
        "turbulent kinetic energy [m2/s2]": (
            turbulent_kinetic_energy_m2_per_s2
        ),
        "turbulent dissipation [m2/s3]": (
            turbulent_dissipation_m2_per_s3
        ),
        "nominal residence time [s]": residence_time_s,
    },
    name="NeqSim-to-OpenFOAM interface",
)


## 5. Build the OpenFOAM 14 case from a maintained tutorial

The case reuses the installed `pitzDailySteady` numerics and turbulence setup, then replaces its
mesh and boundary fields. This avoids silently carrying obsolete solver dictionaries into the
example.


In [ ]:
def run_foam(command, timeout_seconds=900):
    bash_command = (
        f"source {shlex.quote(str(OPENFOAM_BASHRC))} >/dev/null 2>&1 "
        f"&& {command}"
    )
    completed = subprocess.run(
        ["bash", "-lc", bash_command],
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
    )
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(f"OpenFOAM command failed: {command}")
    return completed


def replace_pattern(path, pattern, replacement, flags=0):
    original = path.read_text()
    updated, replacement_count = re.subn(pattern, replacement, original, flags=flags)
    if replacement_count == 0:
        raise RuntimeError(f"Pattern not found in {path}: {pattern}")
    path.write_text(updated)


def foam_field(object_name, field_class, dimensions, internal, boundary_text):
    return f"""FoamFile
{{
    format ascii;
    class {field_class};
    object {object_name};
}}
dimensions {dimensions};
internalField uniform {internal};
boundaryField
{{
{boundary_text}
}}
"""


tutorials_dir = Path(run_foam('printf "%s" "$FOAM_TUTORIALS"').stdout.strip())
tutorial_case = tutorials_dir / "incompressibleFluid" / "pitzDailySteady"
case_dir = runtime_root / "neqsim_openfoam_compressor_inlet_case"

if case_dir.is_dir():
    shutil.rmtree(case_dir)
shutil.copytree(tutorial_case, case_dir)

case_tri_surface = case_dir / "constant" / "triSurface"
case_tri_surface.mkdir(parents=True, exist_ok=True)
for stl_path in tri_surface_dir.glob("*.stl"):
    shutil.copy2(stl_path, case_tri_surface / stl_path.name)

replace_pattern(
    case_dir / "constant" / "physicalProperties",
    r"(?m)^nu\s+[^;]+;",
    f"nu              {kinematic_viscosity_m2_per_s:.9g};",
)

u_boundaries = f"""    inlet
    {{
        type fixedValue;
        value uniform (0 {inlet_velocity_m_per_s:.9g} 0);
    }}
    impellerEye {{ type zeroGradient; }}
    ductWalls {{ type noSlip; }}
    hotCompressorWalls {{ type noSlip; }}"""
p_boundaries = """    inlet { type zeroGradient; }
    impellerEye { type fixedValue; value uniform 0; }
    ductWalls { type zeroGradient; }
    hotCompressorWalls { type zeroGradient; }"""
k_boundaries = f"""    inlet
    {{
        type fixedValue;
        value uniform {turbulent_kinetic_energy_m2_per_s2:.9g};
    }}
    impellerEye {{ type zeroGradient; }}
    ductWalls {{ type kqRWallFunction; value uniform 1e-8; }}
    hotCompressorWalls {{ type kqRWallFunction; value uniform 1e-8; }}"""
epsilon_boundaries = f"""    inlet
    {{
        type fixedValue;
        value uniform {turbulent_dissipation_m2_per_s3:.9g};
    }}
    impellerEye {{ type zeroGradient; }}
    ductWalls {{ type epsilonWallFunction; value uniform 1; }}
    hotCompressorWalls {{ type epsilonWallFunction; value uniform 1; }}"""
nut_boundaries = """    inlet { type calculated; value uniform 0; }
    impellerEye { type calculated; value uniform 0; }
    ductWalls { type nutkWallFunction; value uniform 0; }
    hotCompressorWalls { type nutkWallFunction; value uniform 0; }"""

(case_dir / "0" / "U").write_text(
    foam_field("U", "volVectorField", "[0 1 -1 0 0 0 0]", "(0 0 0)", u_boundaries)
)
(case_dir / "0" / "p").write_text(
    foam_field("p", "volScalarField", "[0 2 -2 0 0 0 0]", "0", p_boundaries)
)
(case_dir / "0" / "k").write_text(
    foam_field(
        "k",
        "volScalarField",
        "[0 2 -2 0 0 0 0]",
        f"{turbulent_kinetic_energy_m2_per_s2:.9g}",
        k_boundaries,
    )
)
(case_dir / "0" / "epsilon").write_text(
    foam_field(
        "epsilon",
        "volScalarField",
        "[0 2 -3 0 0 0 0]",
        f"{turbulent_dissipation_m2_per_s3:.9g}",
        epsilon_boundaries,
    )
)
(case_dir / "0" / "nut").write_text(
    foam_field("nut", "volScalarField", "[0 2 -1 0 0 0 0]", "0", nut_boundaries)
)

(case_dir / "system" / "blockMeshDict").write_text(r"""FoamFile
{
    format ascii;
    class dictionary;
    object blockMeshDict;
}
scale 1;
vertices
(
    (-1.4 -2.8 -0.5) (3.4 -2.8 -0.5)
    (3.4 1.4 -0.5) (-1.4 1.4 -0.5)
    (-1.4 -2.8 0.5) (3.4 -2.8 0.5)
    (3.4 1.4 0.5) (-1.4 1.4 0.5)
);
blocks
(
    hex (0 1 2 3 4 5 6 7) (40 36 10) simpleGrading (1 1 1)
);
edges ();
boundary
(
    background
    {
        type patch;
        faces
        (
            (0 4 7 3) (1 2 6 5) (0 1 5 4)
            (3 7 6 2) (0 3 2 1) (4 5 6 7)
        );
    }
);
mergePatchPairs ();
""")

feature_entries = "\n".join(
    f"""{name}.stl
{{
    extractionMethod extractFromSurface;
    extractFromSurfaceCoeffs {{ includedAngle 150; }}
    writeObj yes;
}}"""
    for name in boundaries
)
(case_dir / "system" / "surfaceFeaturesDict").write_text(feature_entries)

geometry_entries = "\n".join(
    f"""    {name}.stl
    {{
        type triSurfaceMesh;
        name {name};
    }}"""
    for name in boundaries
)
feature_list = "\n".join(
    f'        {{ file "{name}.eMesh"; level 2; }}'
    for name in boundaries
)
refinement_entries = "\n".join(
    f"""        {name}
        {{
            level (2 3);
            patchInfo {{ type {'wall' if 'Walls' in name else 'patch'}; }}
        }}"""
    for name in boundaries
)

snappy_dictionary = f"""FoamFile
{{
    format ascii;
    class dictionary;
    object snappyHexMeshDict;
}}
castellatedMesh true;
snap true;
addLayers false;
geometry
{{
{geometry_entries}
}}
castellatedMeshControls
{{
    maxLocalCells 1000000;
    maxGlobalCells 2500000;
    minRefinementCells 0;
    maxLoadUnbalance 0.10;
    nCellsBetweenLevels 2;
    features
    (
{feature_list}
    );
    refinementSurfaces
    {{
{refinement_entries}
    }}
    resolveFeatureAngle 30;
    refinementRegions {{}}
    locationInMesh (-0.86 -2.0 0.05);
    allowFreeStandingZoneFaces true;
}}
snapControls
{{
    nSmoothPatch 5;
    tolerance 2.0;
    nSolveIter 50;
    nRelaxIter 8;
    nFeatureSnapIter 12;
    implicitFeatureSnap false;
    explicitFeatureSnap true;
    multiRegionFeatureSnap true;
}}
addLayersControls
{{
    relativeSizes true;
    layers {{}}
    expansionRatio 1.2;
    finalLayerThickness 0.3;
    minThickness 0.1;
    nGrow 0;
    featureAngle 60;
    nRelaxIter 5;
    nSmoothSurfaceNormals 1;
    nSmoothNormals 3;
    nSmoothThickness 10;
    maxFaceThicknessRatio 0.5;
    maxThicknessToMedialRatio 0.3;
    minMedialAxisAngle 90;
    nBufferCellsNoExtrude 0;
    nLayerIter 50;
}}
meshQualityControls
{{
    maxNonOrtho 70;
    maxBoundarySkewness 20;
    maxInternalSkewness 4;
    maxConcave 80;
    minVol 1e-15;
    minTetQuality 1e-30;
    minArea -1;
    minTwist 0.02;
    minDeterminant 0.001;
    minFaceWeight 0.02;
    minVolRatio 0.01;
    minTriangleTwist -1;
    nSmoothScale 4;
    errorReduction 0.75;
}}
mergeTolerance 1e-6;
"""
(case_dir / "system" / "snappyHexMeshDict").write_text(snappy_dictionary)

print(f"Runtime OpenFOAM case: {case_dir}")


## 6. Mesh and solve the carrier gas

`snappyHexMesh` retains the internal region identified by `locationInMesh`. `checkMesh` must pass
before the RANS solution is accepted.


In [ ]:
mesh_commands = [
    f"blockMesh -case {shlex.quote(str(case_dir))}",
    f"surfaceFeatures -case {shlex.quote(str(case_dir))}",
    f"snappyHexMesh -overwrite -case {shlex.quote(str(case_dir))}",
    f"checkMesh -allGeometry -allTopology -case {shlex.quote(str(case_dir))}",
]

mesh_logs = []
for command in mesh_commands:
    result = run_foam(command, timeout_seconds=1200)
    mesh_logs.append(result.stdout)

solver_result = run_foam(
    f"foamRun -case {shlex.quote(str(case_dir))}",
    timeout_seconds=1800,
)
(case_dir / "log.foamRun").write_text(solver_result.stdout)

assert "Mesh OK" in mesh_logs[-1]
assert "End" in solver_result.stdout

print("\n".join(solver_result.stdout.splitlines()[-18:]))


## 7. Inspect the impeller-eye flow field

The velocity field is sliced through the bend center plane. The model reports pressure loss and
velocity non-uniformity at the stationary impeller-eye interface.


In [ ]:
conversion_result = run_foam(
    f"foamToVTK -case {shlex.quote(str(case_dir))} -latestTime"
)
latest_time = run_foam(
    f"foamListTimes -case {shlex.quote(str(case_dir))} -latestTime"
).stdout.strip()
vtk_path = case_dir / "VTK" / f"{case_dir.name}_{latest_time}.vtk"

mesh = pv.read(vtk_path)
mesh.point_data["velocity_magnitude"] = np.linalg.norm(
    mesh.point_data["U"],
    axis=1,
)
mesh.point_data["relative_pressure_Pa"] = (
    gas_density_kg_per_m3 * mesh.point_data["p"]
)

center_slice = mesh.slice(normal=(0.0, 0.0, 1.0), origin=(0.0, 0.0, 0.0))
plotter = pv.Plotter(off_screen=True, window_size=(1200, 720))
plotter.add_mesh(
    center_slice,
    scalars="velocity_magnitude",
    cmap="turbo",
    show_edges=False,
    scalar_bar_args={"title": "Velocity [m/s]"},
)
plotter.view_xy()
plotter.camera.zoom(1.25)
image = plotter.screenshot(return_img=True)
plotter.close()

figure, axis = plt.subplots(figsize=(13, 7), constrained_layout=True)
axis.imshow(image)
axis.set_title("OpenFOAM carrier-gas velocity through the compressor inlet")
axis.axis("off")
plt.show()

print(f"Converged time index: {latest_time}")
print(f"Cells: {mesh.n_cells:,}")


In [ ]:
def patch_metrics(patch_name, normal_component):
    patch_path = (
        case_dir / "VTK" / patch_name / f"{patch_name}_{latest_time}.vtk"
    )
    patch = pv.read(patch_path).compute_cell_sizes(
        length=False,
        area=True,
        volume=False,
    )
    areas_m2 = patch.cell_data["Area"]
    velocity = patch.cell_data["U"]
    pressure = patch.cell_data["p"]
    normal_velocity = velocity[:, normal_component]
    mass_flow = gas_density_kg_per_m3 * abs(np.sum(normal_velocity * areas_m2))
    area_average_pressure = np.average(pressure, weights=areas_m2)
    speed_squared = np.sum(velocity**2, axis=1)
    total_pressure = gas_density_kg_per_m3 * np.average(
        pressure + 0.5 * speed_squared,
        weights=areas_m2,
    )
    return {
        "area [m2]": float(np.sum(areas_m2)),
        "mass flow [kg/s]": float(mass_flow),
        "static pressure [Pa rel.]": (
            gas_density_kg_per_m3 * area_average_pressure
        ),
        "total pressure [Pa rel.]": float(total_pressure),
        "velocity coefficient of variation [-]": float(
            np.std(np.linalg.norm(velocity, axis=1))
            / np.mean(np.linalg.norm(velocity, axis=1))
        ),
    }


inlet_metrics = patch_metrics("inlet", 1)
eye_metrics = patch_metrics("impellerEye", 0)
mass_balance_error_percent = 100.0 * abs(
    eye_metrics["mass flow [kg/s]"] - inlet_metrics["mass flow [kg/s]"]
) / inlet_metrics["mass flow [kg/s]"]
total_pressure_loss_pa = (
    inlet_metrics["total pressure [Pa rel.]"]
    - eye_metrics["total pressure [Pa rel.]"]
)

cfd_validation = pd.Series(
    {
        "inlet mass flow [kg/s]": inlet_metrics["mass flow [kg/s]"],
        "eye mass flow [kg/s]": eye_metrics["mass flow [kg/s]"],
        "mass-balance error [%]": mass_balance_error_percent,
        "total-pressure loss [Pa]": total_pressure_loss_pa,
        "eye velocity coefficient of variation [-]": (
            eye_metrics["velocity coefficient of variation [-]"]
        ),
    },
    name="Carrier-flow validation",
)

assert mass_balance_error_percent < 2.0
assert total_pressure_loss_pa > 0.0
cfd_validation


## 8. Add liquid inertia and the hot-running compressor

For a first screening, the bend response is represented by

$$
\mathrm{Stk}=\frac{\rho_l d^2 U}{18\mu_g R_b}.
$$

Fine droplets with $\mathrm{Stk}<0.1$ tend to follow the gas. Values near unity show partial
impaction, while large values indicate strong inertial impaction.

Hot-contact evaporation uses the explicit sensitivity

$$
d^2(t)=d_0^2-Kt.
$$

The value of $K$ is not silently fitted. It is an editable screening input and must be replaced
by an OpenFOAM reacting parcel model or a NeqSim-based multicomponent Maxwell–Stefan model when
evaporation distance is a design result.


In [ ]:
DROPLET_DIAMETERS_UM = np.array([50.0, 150.0, 500.0])
HOT_WALL_TEMPERATURE_C = 110.0
HOT_CONTACT_TIME_S = 0.25
COLD_EVAPORATION_CONSTANT_M2_PER_S = 1.0e-9
HOT_EVAPORATION_CONSTANT_M2_PER_S = 3.0e-8


def evaporated_mass_fraction(diameter_m, evaporation_constant, time_s):
    remaining_diameter_squared = max(
        diameter_m**2 - evaporation_constant * time_s,
        0.0,
    )
    remaining_diameter = math.sqrt(remaining_diameter_squared)
    return 1.0 - (remaining_diameter / diameter_m) ** 3


screening_rows = []
for diameter_um in DROPLET_DIAMETERS_UM:
    diameter_m = diameter_um * 1.0e-6
    relaxation_time_s = (
        liquid_density_kg_per_m3
        * diameter_m**2
        / (18.0 * gas_viscosity_pa_s)
    )
    stokes_number = (
        relaxation_time_s * inlet_velocity_m_per_s / BEND_RADIUS_M
    )
    if stokes_number < 0.1:
        inertia_regime = "follows gas"
    elif stokes_number < 1.0:
        inertia_regime = "partial impaction"
    else:
        inertia_regime = "strong impaction"

    screening_rows.append(
        {
            "diameter [µm]": diameter_um,
            "relaxation time [s]": relaxation_time_s,
            "bend Stokes number [-]": stokes_number,
            "inertia regime": inertia_regime,
            "cold free-flight evaporation [%]": 100.0
            * evaporated_mass_fraction(
                diameter_m,
                COLD_EVAPORATION_CONSTANT_M2_PER_S,
                residence_time_s,
            ),
            "hot-contact evaporation [%]": 100.0
            * evaporated_mass_fraction(
                diameter_m,
                HOT_EVAPORATION_CONSTANT_M2_PER_S,
                HOT_CONTACT_TIME_S,
            ),
        }
    )

screening_table = pd.DataFrame(screening_rows)
screening_table


In [ ]:
gas_cp_j_per_kgk = gas_phase.getCp("J/kgK")
gas_conductivity_w_per_mk = gas_phase.getThermalConductivity("W/mK")
prandtl_number = (
    gas_cp_j_per_kgk
    * gas_viscosity_pa_s
    / gas_conductivity_w_per_mk
)
nusselt_number = 0.023 * reynolds_number**0.8 * prandtl_number**0.4
heat_transfer_coefficient_w_per_m2k = (
    nusselt_number * gas_conductivity_w_per_mk / INLET_DIAMETER_M
)
hot_length_m = (1.0 - HOT_WALL_START_FRACTION) * centerline_length_m
hot_area_m2 = math.pi * INLET_DIAMETER_M * hot_length_m
heat_capacity_rate_w_per_k = gas_mass_flow_kg_per_s * gas_cp_j_per_kgk
number_transfer_units = (
    heat_transfer_coefficient_w_per_m2k
    * hot_area_m2
    / heat_capacity_rate_w_per_k
)
inlet_temperature_c = TEMPERATURE_K - 273.15
gas_eye_temperature_c = HOT_WALL_TEMPERATURE_C - (
    HOT_WALL_TEMPERATURE_C - inlet_temperature_c
) * math.exp(-number_transfer_units)

figure, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
axes[0].plot(
    screening_table["diameter [µm]"],
    screening_table["bend Stokes number [-]"],
    marker="s",
    color="#756bb1",
)
axes[0].axhline(0.1, color="gray", linestyle="--")
axes[0].axhline(1.0, color="black", linestyle="--")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Droplet diameter [µm]")
axes[0].set_ylabel("Bend Stokes number [-]")

axes[1].plot(
    screening_table["diameter [µm]"],
    screening_table["hot-contact evaporation [%]"],
    marker="o",
    color="#e6550d",
)
axes[1].set_xscale("log")
axes[1].set_ylim(0.0, 105.0)
axes[1].set_xlabel("Droplet diameter [µm]")
axes[1].set_ylabel("Hot-contact evaporation sensitivity [% mass]")

figure.suptitle("Wet-gas compressor-inlet screening")
plt.show()

print(f"Estimated bulk-gas temperature at eye: {gas_eye_temperature_c:.2f} °C")


## Engineering interpretation

- The OpenFOAM carrier solution quantifies total-pressure loss and velocity distortion created
  by the bend and contraction.
- Fine droplets are more likely to follow the gas and respond to hot-surface evaporation.
- Large droplets retain momentum, have a stronger outer-bend impaction tendency, and evaporate
  only weakly during short contact.
- At high gas flow, the hot wall may warm the bulk gas only slightly. Local wall contact can
  therefore matter more than average gas heating.
- A high Stokes number is an impaction indicator, not a deposition fraction. The next fidelity
  level should track parcels in the solved CFD velocity field and use a calibrated wall model.


## Limitations and recommended extension

This notebook is a concept and teaching model. It does not certify liquid tolerance for a
compressor.

Before engineering use:

1. replace the simplified inlet with vendor CAD, including the inlet cone and impeller eye;
2. run mesh and turbulence-model independence studies;
3. replace fixed carryover with scrubber performance and measured droplet-size data;
4. use a version-matched `reactingParcelFoam`/spray solver for parcel trajectories, breakup,
   deposition, film formation, heat transfer, and evaporation;
5. use NeqSim non-equilibrium multicomponent mass transfer when condensate composition changes
   materially during evaporation;
6. include rotating impeller and volute geometry when compression heating and blade interaction
   are in scope;
7. validate against pressure-drop data, liquid carryover measurements, and compressor-vendor
   operating limits.

The notebook uses SI units, absolute pressure for the thermodynamic state, and reports all
screening assumptions explicitly.


## Reproducibility

- NeqSim Python: 3.16.0
- OpenFOAM: Foundation release 14
- PyVista: 0.48.4
- EOS: SRK with classic mixing rule
- Geometry: generated from dimensions in this notebook
- Evidence level: topology and conservation checks plus analytical trend screening; no vendor or
  experimental compressor benchmark is claimed

References:

- [NeqSim documentation](https://equinor.github.io/neqsim/)
- [OpenFOAM documentation](https://doc.cfd.direct/openfoam/user-guide-v14/contents)
- [OpenFOAM mesh utilities](https://doc.cfd.direct/openfoam/user-guide-v14/standard-utilities)
